**training: 80% web data and 20% wild data**

**test: Web**




In [ ]:
#Dataset Base Path
base_path = "/datasetfolderpath/anonymous/lote-animal-mix-s3"

train_image_dir = f"{base_path}/train/images"
train_label_dir = f"{base_path}/train/labels"
val_image_dir   = f"{base_path}/val/images"
val_label_dir   = f"{base_path}/val/labels"
test_image_dir  = f"{base_path}/test/images"
test_label_dir  = f"{base_path}/test/labels"

# YAML Save Paths
yaml_path = "/datasetfolderpath/anonymous/s3_yolo26/working/dataset.yaml"

# Trained Model Path
best_model_path = "/datasetfolderpath/anonymous/s3_yolo26/working/runs/detect/train/weights/best.pt"

In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install ultralytics opencv-python-headless matplotlib pandas

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO
from ultralytics import RTDETR

In [ ]:
# Verify Dataset Structure
print("\nVerifying dataset structure")

print("Dataset Structure:")
print("-" * 20)

directories_to_check = [
    ('Train Images', train_image_dir),
    ('Train Labels', train_label_dir),
    ('Val Images', val_image_dir),
    ('Val Labels', val_label_dir),
    ('Test Images', test_image_dir),
    ('Test Labels', test_label_dir)
]

for name, dir_path in directories_to_check:
    if os.path.exists(dir_path):
        files = [f for f in os.listdir(dir_path) if f.endswith(('.jpg', '.png', '.jpeg', '.txt'))]
        file_type = "images" if 'image' in name.lower() else "labels"
        print(f"{name}: {len(files)} {file_type}")
        if files:
            print(f"   Sample: {files[0]}")
    else:
        print(f"{name}: Directory not found - {dir_path}")

# Check label format and classes
print("\nChecking label format and classes...")
if os.path.exists(train_label_dir):
    label_files = [f for f in os.listdir(train_label_dir) if f.endswith('.txt')]
    if label_files:
        sample_label_path = os.path.join(train_label_dir, label_files[0])
        with open(sample_label_path, 'r') as f:
            sample_content = f.readlines()
        if sample_content:
            print(f" Sample label format: {sample_content[0].strip()}")
        
        class_ids = set()
        for label_file in label_files[:20]: 
            with open(os.path.join(train_label_dir, label_file), 'r') as f:
                for line in f:
                    if line.strip():
                        try:
                            class_id = int(line.strip().split()[0])
                            class_ids.add(class_id)
                        except:
                            continue
        
        print(f" Classes found in labels: {sorted(class_ids)}")
        num_classes = max(class_ids) + 1 if class_ids else 11
        print(f" Number of classes: {num_classes}")
    else:
        num_classes = 11
        print("  No label files found, using default: 11 classes")
else:
    num_classes = 11
    print("  Labels directory not found, using default: 11 classes")

In [ ]:
# Create Dataset YAML
print("\nCreating dataset YAML configuration...")

class_names = [
    'Pandas','RedPandas','MartesFlavigula','MacacaThibetana',
    'RhinopithecusRoxellana','HystrixBrachyura','SusScrofa',
    'RusaUnicolor','ElaphodusCephalophus','CapricornisMilneedwardsii',
    'PseudoisNayaur'
]

dataset_yaml = f"""

path: {base_path}
train: train/images
val: val/images
test: test/images

nc: 11
names: {class_names}
"""

with open(yaml_path, 'w') as f:
    f.write(dataset_yaml)
print(" Dataset YAML created:", yaml_path)

print("\nDataset YAML Content:")
print("-" * 20)
print(dataset_yaml)

In [ ]:
# TRAINING CHECKPOINT HANDLING
resume_path = "/datasetfolderpath/anonymous/s3_yolo26/working/runs/detect/train/weights/last.pt"

if os.path.exists(resume_path):
    print("\nCheckpoint found! Resuming training from last.pt")
    model = YOLO(resume_path)
    resume_flag = True
else:
    print("\nNo checkpoint found. Starting fresh training!")
    model = YOLO("yolo26l.pt")
    resume_flag = False

In [ ]:
results = model.train(
    data=yaml_path,
    epochs=500,
    imgsz=800,
    batch=8,
    device=0,
    workers=4,
    patience=30,
    save=True,
    pretrained=True,
    optimizer='auto',
    lr0=0.001,
    weight_decay=0.0005,
    exist_ok=True,
    verbose=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.001,
    flipud=0.01,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.1,
    copy_paste=0.1,
    resume=resume_flag    
)


print("\nTraining Complete!")
if resume_flag:
    print("Training continued from previous checkpoint.")
else:
    print("Training started fresh.")

In [ ]:
print("\nModel validation on test set!")
print("-" * 20)

model = YOLO(best_model_path)

test_metrics = model.val(
    data=yaml_path,
    split='test',
    imgsz=800
)

print("\nValidation Results:")
print(f"mAP50:    {test_metrics.box.map50:.4f}")
print(f"mAP50-95: {test_metrics.box.map:.4f}")
print(f"Precision: {test_metrics.box.p.mean():.4f}")
print(f"Recall:    {test_metrics.box.r.mean():.4f}")